In [1]:
import os
import re
import sqlite3

DUMP = "../../../data/northwind.sql"
DB = "northwind.sqlite"

with open(DUMP) as f:
    sql = f.read()

sql = re.sub(r"^SET .*?;\s*$", "", sql, flags=re.M)
sql = re.sub(r"ALTER TABLE ONLY[^;]*;", "", sql, flags=re.S)
sql = sql.replace("bytea", "BLOB")
sql = sql.replace(r"'\x'", "NULL")

if os.path.exists(DB):
    os.remove(DB)
conn = sqlite3.connect(DB)
conn.executescript(sql)
conn.commit()

cur = conn.cursor()
for table in ["customers", "orders", "order_details", "products", "employees"]:
    cur.execute(f"SELECT COUNT(*) FROM {table}")
    print(f"{table}: {cur.fetchone()[0]}")

customers: 91
orders: 830
order_details: 2155
products: 77
employees: 9


# Exercice 1: Sélection avec conditions multiples
- Afficher tous les order_id et ship_name pour les commandes expédiées en Suisse (ship_country = 'Switzerland') ou en Argentine (ship_country = 'Argentina'), avec un freight supérieur à 20.


In [2]:
# 1. On ouvre la connexion
conn = sqlite3.connect("northwind.sqlite")
cur = conn.cursor()

# 2. Écriture et exécution de la requête SQL
query = """
SELECT order_id, ship_name, ship_country, freight
FROM orders
WHERE ship_country IN ('Switzerland', 'Argentina')
  AND freight > 20;
"""

cur.execute(query)
results = cur.fetchall()

# affichage 
for row in results:
    print(row)

# 4. Fermeture de la connexion
conn.close()

(10254, 'Chop-suey Chinese', 'Switzerland', 22.9799995)
(10255, 'Richter Supermarkt', 'Switzerland', 148.330002)
(10409, 'OcÃ©ano AtlÃ¡ntico Ltda.', 'Argentina', 29.8299999)
(10419, 'Richter Supermarkt', 'Switzerland', 137.350006)
(10448, 'Rancho grande', 'Argentina', 38.8199997)
(10519, 'Chop-suey Chinese', 'Switzerland', 91.7600021)
(10537, 'Richter Supermarkt', 'Switzerland', 78.8499985)
(10666, 'Richter Supermarkt', 'Switzerland', 232.419998)
(10716, 'Rancho grande', 'Argentina', 22.5699997)
(10731, 'Chop-suey Chinese', 'Switzerland', 96.6500015)
(10746, 'Chop-suey Chinese', 'Switzerland', 31.4300003)
(10751, 'Richter Supermarkt', 'Switzerland', 130.789993)
(10758, 'Richter Supermarkt', 'Switzerland', 138.169998)
(10828, 'Rancho grande', 'Argentina', 90.8499985)
(10916, 'Rancho grande', 'Argentina', 63.7700005)
(10937, 'Cactus Comidas para llevar', 'Argentina', 31.5100002)
(10951, 'Richter Supermarkt', 'Switzerland', 30.8500004)
(10958, 'OcÃ©ano AtlÃ¡ntico Ltda.', 'Argentina', 49.5

# Exercice 2: Tri et limite sur sélection filtrée

-  Trouver les 5 commandes ayant le plus haut freight, expédiées en 1997 (order_date BETWEEN '1997-01-01' AND '1997-12-31').

In [3]:
# 1. Connexion à la base de données
conn = sqlite3.connect("northwind.sqlite")
cur = conn.cursor()

# 2. Requête SQL
query = """
SELECT *
FROM orders
WHERE order_date BETWEEN '1997-01-01' AND '1997-12-31'
ORDER BY freight DESC
LIMIT 5;
"""

cur.execute(query)
results = cur.fetchall()

# 3. Affichage simple
for row in results:
    print(row)

# 4. Fermeture de la connexion
conn.close()

(10540, 'QUICK', 3, '1997-05-19', '1997-06-16', '1997-06-13', 3, 1007.64001, 'QUICK-Stop', 'TaucherstraÃŸe 10', 'Cunewalde', None, '01307', 'Germany')
(10691, 'QUICK', 2, '1997-10-03', '1997-11-14', '1997-10-22', 2, 810.049988, 'QUICK-Stop', 'TaucherstraÃŸe 10', 'Cunewalde', None, '01307', 'Germany')
(10514, 'ERNSH', 3, '1997-04-22', '1997-05-20', '1997-05-16', 2, 789.950012, 'Ernst Handel', 'Kirchgasse 6', 'Graz', None, '8010', 'Austria')
(10479, 'RATTC', 3, '1997-03-19', '1997-04-16', '1997-03-21', 3, 708.950012, 'Rattlesnake Canyon Grocery', '2817 Milton Dr.', 'Albuquerque', 'NM', '87110', 'USA')
(10612, 'SAVEA', 1, '1997-07-28', '1997-08-25', '1997-08-01', 2, 544.080017, 'Save-a-lot Markets', '187 Suffolk Ln.', 'Boise', 'ID', '83720', 'USA')


# Exercice 3: Utilisation de DISTINCT
- Lister les combinaisons uniques de ship_country et ship_city pour toutes les commandes.


In [4]:
# 1. Connexion à la base de données
conn = sqlite3.connect("northwind.sqlite")
cur = conn.cursor()

# 2. Requête SQL avec DISTINCT
query = """
SELECT DISTINCT ship_country, ship_city
FROM orders
LIMIT 10;
"""

cur.execute(query)
results = cur.fetchall()

# 3. Affichage simple
for row in results:
    print(row)

# 4. Fermeture de la connexion
conn.close()

('France', 'Reims')
('Germany', 'MÃ¼nster')
('Brazil', 'Rio de Janeiro')
('France', 'Lyon')
('Belgium', 'Charleroi')
('Switzerland', 'Bern')
('Switzerland', 'GenÃ¨ve')
('Brazil', 'Resende')
('Venezuela', 'San CristÃ³bal')
('Austria', 'Graz')


# Exercice 4: Combiner WHERE, ORDER BY et LIMIT
- Afficher les 3 premières commandes (basées sur la order_date) pour les clients en Brésil (ship_country = 'Brazil').

In [5]:
# 1. Connexion à la base de données
conn = sqlite3.connect("northwind.sqlite")
cur = conn.cursor()

# 2. Requête SQL
query = """
SELECT *
FROM orders
WHERE ship_country = 'Brazil'
ORDER BY order_date ASC
LIMIT 3;
"""

cur.execute(query)
results = cur.fetchall()

# 3. Affichage simple
for row in results:
    print(row)

# 4. Fermeture de la connexion
conn.close()

(10250, 'HANAR', 4, '1996-07-08', '1996-08-05', '1996-07-12', 2, 65.8300018, 'Hanari Carnes', 'Rua do PaÃ§o, 67', 'Rio de Janeiro', 'RJ', '05454-876', 'Brazil')
(10253, 'HANAR', 3, '1996-07-10', '1996-07-24', '1996-07-16', 2, 58.1699982, 'Hanari Carnes', 'Rua do PaÃ§o, 67', 'Rio de Janeiro', 'RJ', '05454-876', 'Brazil')
(10256, 'WELLI', 3, '1996-07-15', '1996-08-12', '1996-07-17', 2, 13.9700003, 'Wellington Importadora', 'Rua do Mercado, 12', 'Resende', 'SP', '08737-363', 'Brazil')



# Exercice 5: Gestion des NULL avec tri et filtrage

- Sélectionner toutes les commandes où ship_region est NULL, en les triant par shipped_date en ordre décroissant.


In [6]:
# 1. Connexion à la base de données
conn = sqlite3.connect("northwind.sqlite")
cur = conn.cursor()

# 2. Requête SQL
query = """
SELECT order_id, ship_name, ship_region, shipped_date
FROM orders
WHERE ship_region IS NULL
ORDER BY shipped_date DESC
LIMIT 10;
"""

cur.execute(query)
results = cur.fetchall()

# 3. Affichage simple
for row in results:
    print(row)

# 4. Fermeture de la connexion
conn.close()

(11067, 'Drachenblut Delikatessen', None, '1998-05-06')
(11069, 'Tortuga Restaurante', None, '1998-05-06')
(11050, 'Folk och fÃ¤ HB', None, '1998-05-05')
(11060, 'Franchi S.p.A.', None, '1998-05-04')
(11044, 'Wolski Zajazd', None, '1998-05-01')
(11047, 'Eastern Connection', None, '1998-05-01')
(11056, 'Eastern Connection', None, '1998-05-01')
(11057, 'North/South', None, '1998-05-01')
(11038, 'SuprÃªmes dÃ©lices', None, '1998-04-30')
(11043, 'SpÃ©cialitÃ©s du monde', None, '1998-04-29')



# Exercice 6: Utilisation de l'opérateur LIKE
- Trouver les commandes dont le nom du destinataire (ship_name) commence par 'S'.

In [7]:
# 1. Connexion à la base de données
conn = sqlite3.connect("northwind.sqlite")
cur = conn.cursor()

# 2. Requête SQL avec LIKE
query = """
SELECT order_id, ship_name
FROM orders
WHERE ship_name LIKE 'S%';
"""

cur.execute(query)
results = cur.fetchall()

# 3. Affichage simple
for row in results:
    print(row)

# 4. Fermeture de la connexion
conn.close()

(10252, 'SuprÃªmes dÃ©lices')
(10271, 'Split Rail Beer & Ale')
(10302, 'SuprÃªmes dÃ©lices')
(10324, 'Save-a-lot Markets')
(10329, 'Split Rail Beer & Ale')
(10341, 'Simons bistro')
(10349, 'Split Rail Beer & Ale')
(10359, 'Seven Seas Imports')
(10369, 'Split Rail Beer & Ale')
(10377, 'Seven Seas Imports')
(10385, 'Split Rail Beer & Ale')
(10387, 'SantÃ© Gourmet')
(10388, 'Seven Seas Imports')
(10393, 'Save-a-lot Markets')
(10398, 'Save-a-lot Markets')
(10417, 'Simons bistro')
(10432, 'Split Rail Beer & Ale')
(10440, 'Save-a-lot Markets')
(10452, 'Save-a-lot Markets')
(10458, 'SuprÃªmes dÃ©lices')
(10463, 'SuprÃªmes dÃ©lices')
(10472, 'Seven Seas Imports')
(10475, 'SuprÃªmes dÃ©lices')
(10510, 'Save-a-lot Markets')
(10520, 'SantÃ© Gourmet')
(10523, 'Seven Seas Imports')
(10547, 'Seven Seas Imports')
(10555, 'Save-a-lot Markets')
(10556, 'Simons bistro')
(10603, 'Save-a-lot Markets')
(10607, 'Save-a-lot Markets')
(10612, 'Save-a-lot Markets')
(10627, 'Save-a-lot Markets')
(10639, 'SantÃ©

# Exercice 7: Plusieurs conditions avec AND, OR
- Sélectionner les commandes expédiées en 'UK' ou 'USA' avec un freight supérieur à 50, et passées avant l'année 1998.

In [8]:
# 1. Connexion à la base de données
conn = sqlite3.connect("northwind.sqlite")
cur = conn.cursor()

# 2. Requête SQL
query = """
SELECT order_id, ship_country, freight, order_date
FROM orders
WHERE ship_country IN ('UK', 'USA')
  AND freight > 50
  AND order_date < '1998-01-01';
"""

cur.execute(query)
results = cur.fetchall()

# 3. Affichage simple
for row in results:
    print(row)

# 4. Fermeture de la connexion
conn.close()

(10272, 'USA', 98.0299988, '1996-08-02')
(10294, 'USA', 147.259995, '1996-08-30')
(10305, 'USA', 257.619995, '1996-09-13')
(10314, 'USA', 74.1600037, '1996-09-25')
(10316, 'USA', 150.149994, '1996-09-27')
(10324, 'USA', 214.270004, '1996-10-08')
(10329, 'USA', 191.669998, '1996-10-15')
(10338, 'USA', 84.2099991, '1996-10-25')
(10346, 'USA', 142.080002, '1996-11-05')
(10359, 'UK', 288.429993, '1996-11-21')
(10364, 'UK', 71.9700012, '1996-11-26')
(10369, 'USA', 195.679993, '1996-12-02')
(10393, 'USA', 126.559998, '1996-12-25')
(10398, 'USA', 89.1600037, '1996-12-30')
(10400, 'UK', 83.9300003, '1997-01-01')
(10440, 'USA', 86.5299988, '1997-02-10')
(10441, 'USA', 73.0199966, '1997-02-10')
(10452, 'USA', 140.259995, '1997-02-20')
(10469, 'USA', 60.1800003, '1997-03-10')
(10479, 'USA', 708.950012, '1997-03-19')
(10504, 'USA', 59.1300011, '1997-04-11')
(10510, 'USA', 367.630005, '1997-04-18')
(10523, 'UK', 77.6299973, '1997-05-01')
(10532, 'UK', 74.4599991, '1997-05-09')
(10547, 'UK', 178.429

# Exercice 8: Requête plusieurs clauses
 - Afficher les order_id, la date de commande (order_date), et le nom du destinataire (ship_name) pour les 10 premières commandes passées après le 1er janvier 1997, où la région de livraison (ship_region) est inconnue (NULL), et le pays de livraison (ship_country) n'est ni 'USA' ni 'UK'. Utiliser un alias pour renommer order_date en DateOfOrder.

In [9]:

# 1. Connexion à la base de données
conn = sqlite3.connect("northwind.sqlite")
cur = conn.cursor()

# 2. Requête SQL complète
query = """
SELECT order_id, order_date AS DateOfOrder, ship_name
FROM orders
WHERE order_date > '1997-01-01'
  AND ship_region IS NULL
  AND ship_country NOT IN ('USA', 'UK')
ORDER BY order_date ASC
LIMIT 10;
"""

cur.execute(query)
results = cur.fetchall()

# 3. Affichage simple
for row in results:
    print(row)

# 4. Fermeture de la connexion
conn.close()

 





(10402, '1997-01-02', 'Ernst Handel')
(10403, '1997-01-03', 'Ernst Handel')
(10404, '1997-01-03', 'Magazzini Alimentari Riuniti')
(10407, '1997-01-07', 'Ottilies KÃ¤seladen')
(10408, '1997-01-08', 'Folies gourmandes')
(10409, '1997-01-09', 'OcÃ©ano AtlÃ¡ntico Ltda.')
(10412, '1997-01-13', 'Wartian Herkku')
(10413, '1997-01-14', "La maison d'Asie")
(10416, '1997-01-16', 'Wartian Herkku')
(10417, '1997-01-16', 'Simons bistro')


# 2 jointures à gauche et préciser à quoi elles pourrait servir

# Exemple 1

In [10]:
# 1. Connexion à la base de données
conn = sqlite3.connect("northwind.sqlite")
cur = conn.cursor()

# On prend TOUS les clients (à gauche) et on y colle leurs commandes (à droite)
query = """

SELECT customers.customer_id, customers.company_name, orders.order_id
FROM customers
LEFT JOIN orders ON customers.customer_id = orders.customer_id;
"""

cur.execute(query)
results = cur.fetchall()

# 3. Affichage simple
for row in results[:10]:
    print(row)

# 4. Fermeture de la connexion
conn.close()


('ALFKI', 'Alfreds Futterkiste', 10643)
('ALFKI', 'Alfreds Futterkiste', 10692)
('ALFKI', 'Alfreds Futterkiste', 10702)
('ALFKI', 'Alfreds Futterkiste', 10835)
('ALFKI', 'Alfreds Futterkiste', 10952)
('ALFKI', 'Alfreds Futterkiste', 11011)
('ANATR', 'Ana Trujillo Emparedados y helados', 10308)
('ANATR', 'Ana Trujillo Emparedados y helados', 10625)
('ANATR', 'Ana Trujillo Emparedados y helados', 10759)
('ANATR', 'Ana Trujillo Emparedados y helados', 10926)


### Ça sert à faire une analyse commerciale pour identifier quels clients sont actifs et, surtout, quels clients n'ont jamais passé de commande (pour pouvoir les relancer par email, par exemple).

# Exemple 2

In [11]:
conn = sqlite3.connect("northwind.sqlite")
cur = conn.cursor()

# On prend TOUS les employés et on compte leurs commandes
query = """
SELECT employees.employee_id, employees.last_name, COUNT(orders.order_id) AS total_ventes
FROM employees
LEFT JOIN orders ON employees.employee_id = orders.employee_id
GROUP BY employees.employee_id;
"""

cur.execute(query)
results = cur.fetchall()

for row in results:
    print(row)

conn.close()

(1, 'Davolio', 123)
(2, 'Fuller', 96)
(3, 'Leverling', 127)
(4, 'Peacock', 156)
(5, 'Buchanan', 42)
(6, 'Suyama', 67)
(7, 'King', 72)
(8, 'Callahan', 104)
(9, 'Dodsworth', 43)


### Ça sert à suivre la performance de l'équipe de vente. En utilisant un LEFT JOIN, on s'assure d'afficher tous les employés de l'entreprise, même une nouvelle recrue qui vient d'arriver et qui n'a pas encore enregistré sa première vente (elle affichera un score de 0 au lieu de disparaître du classement).

In [12]:
conn = sqlite3.connect("northwind.sqlite")
cur = conn.cursor()

# 1. Itération directe sur le curseur
cur.execute("""
    SELECT c.company_name, o.order_id, o.order_date
    FROM customers c
    LEFT JOIN orders o ON c.customer_id = o.customer_id
    LIMIT 10
""")

for row in cur:
    print(row)

('Alfreds Futterkiste', 10643, '1997-08-25')
('Alfreds Futterkiste', 10692, '1997-10-03')
('Alfreds Futterkiste', 10702, '1997-10-13')
('Alfreds Futterkiste', 10835, '1998-01-15')
('Alfreds Futterkiste', 10952, '1998-03-16')
('Alfreds Futterkiste', 11011, '1998-04-09')
('Ana Trujillo Emparedados y helados', 10308, '1996-09-18')
('Ana Trujillo Emparedados y helados', 10625, '1997-08-08')
('Ana Trujillo Emparedados y helados', 10759, '1997-11-28')
('Ana Trujillo Emparedados y helados', 10926, '1998-03-04')


### ça sert à créer la liste de tes clients et à voir immédiatement s'ils ont commandé ou non.